# Managing data

ASCOT5 can model a wide range of physical phenomena in a variety of configurations.
To support this flexibility, the code uses several different input types.
Inputs are organized into *categories*, and each category can have one or more *variants*.

For example, `bfield` is the category that represents magnetic field inputs.
A category can be thought of as an interface that defines what information is required, while a variant provides a specific implementation of that interface.
For instance, `BfieldSpline2D` represents an axisymmetric tokamak magnetic field interpolated with cubic splines, whereas `BfieldStellarator` represents a stellarator magnetic field.

Managing inputs and outputs is therefore somewhat involved, which is why a dedicated tutorial is warranted.
The reason for the apparent complexity is the design that is guided by several principles:

- Simulation outputs should always be stored together with the inputs that produced them.
  This guarantees that results remain reproducible.
- Parameter scans should be supported naturally, meaning that multiple input datasets and multiple simulation outputs can coexist in the same data storage.
- ASCOT5 should be able to operate independently of how data is stored, allowing the same interface to be used whether data resides on disk or is managed in memory.

Furthermore, some datasets can be very large, particularly magnetic field data, wall geometries, and simulation outputs.
When data is stored on disk, ASCOT5 uses the HDF5 format, which is a widely used modern format designed for scientific data.

Almost everything in ASCOT5 is accessed through the `Ascot` object, so we begin by inititalizing one.

In [1]:
from a5py import Ascot
a5 = Ascot()


The `Ascot` object has `data` attribute that manages the data.
To get an overview of what is stored, use the `show_contents()` method.

In [2]:
a5.data.show_contents()

Inputs: [only active shown]
wall      *no inputs*

mhd       *no inputs*

bfield    *no inputs*

plasma    *no inputs*

marker    *no inputs*

boozer    *no inputs*

neutral   *no inputs*

efield    *no inputs*

nbi       *no inputs*

asigma    *no inputs*


Simulations:
No simulation results.



Creating inputs and executing simulations was already covered in the introduction.
Let's use the same test case here to generate some data:

In [3]:
import unyt
import numpy as np
from a5py import SimulationOptions
from a5py.templates import PremadeMagneticField

opt = SimulationOptions.from_dict(
    simulation={"mode": "gyro-orbit", "timestep": 1e-8},
    physics={"enable_orbit_following": True},
    endconditions={"activate_simulation_time_limits": True, "max_mileage": 1e-4,},
)

template = PremadeMagneticField(a5, field="iter-baseline")
template.create_input()

nmrk = 1
a5.data.create_guidingcentermarker(
    species="alpha",
    r=(6.2 + (8. - 6.2) * np.random.rand(nmrk))*unyt.m,
    z=0.*unyt.m,
    ekin= 3.5e6*unyt.eV,
    pitch=1. - 2 * np.random.rand(nmrk),
)

run = a5.simulate(params=opt)
a5.data.show_contents()

Inputs: [only active shown]
wall      *no inputs*

mhd       *no inputs*

bfield    BfieldAnalytical_1 2026-06-23 13:33:55 (no other inputs)
               ""
plasma    *no inputs*

marker    GuidingcenterMarker_1 2026-06-23 13:33:55 (no other inputs)
               ""
boozer    *no inputs*

neutral   *no inputs*

efield    *no inputs*

nbi       *no inputs*

asigma    *no inputs*


Simulations:
Run_1           2026-06-23 13:33:55 [active]
               ""



Note that this displays only a single input within each category.
To display all inputs within a category, use the corresponding `show_contents` method:

In [5]:
a5.data.bfield.show_contents()

BfieldAnalytical_1 2026-06-23 13:33:55 [active]
<no tag>





## The active status

One input in each category and one output is always marked as *active*.
The active dataset can be accessed from the corresponding category (outputs are stored on the top level in the tree hierarchy):

In [6]:
print(a5.data.bfield.active)
print(a5.data.active)

<BfieldAnalytical(name=BfieldAnalytical_1, date=2026-06-23 13:33:55)>
<Run(name=Run_1, inputs=['(bfield:BfieldAnalytical_1)', '(marker:GuidingcenterMarker_1)'], diagnostics=['endstate'], saved=False)>


**The active status means that this dataset will be used in all routines unless other dataset is given explicitly**.
Usually the methods provide parameters to override the active dataset.
The active status is stored in file.
Whenever a simulation is run, the most recent output is set as active.
However, for inputs the active input is not changed by default whenever a new input is created.

The active dataset can be changed using the `activate` method.


In [7]:
a5.data.bfield.active.activate() # Does nothing since the input is already active

## Date and note

All datasets store the `date` when they were created (this cannot be modified by the user):

In [8]:
a5.data.bfield.active.date

'2026-06-23 13:33:55'

The data can be documented using the `note` property which is mutable.

In [9]:
print(a5.data.bfield.active.note)
a5.data.active.note = "This is the active field."
print(a5.data.active.note)


This is the active field.


## Names and tags

To support multiple inputs and outputs, all of them are assigned an unique *name* which has the format `<name of the variant>_<running index>`.
For example, here we have `BfieldAnalytical_1`, `GuidingcenterMarker_1`, and `Run_1`.
These are used to reference and access the data within the corresponding category (outputs are stored on the top level in the tree hierarchy):

In [10]:
print(a5.data.bfield.BfieldAnalytical_1.name)
print(a5.data.Run_1.name)

BfieldAnalytical_1
Run_1


The name is immutable to ensure that each dataset has a unique name.
For better user-experience, there's a way to specify a *tag* which can be used in place of the name.
The tag is set via the note attribute: 


In [42]:
a5.data.bfield.active.note = "My <new> field"
a5.data.bfield.NEW.note

'My <NEW> field'

The tag may also contain multiple words but these are then parsed when forming a tag:

In [43]:
a5.data.bfield.NEW.note = "<Even better tag>"
a5.data.bfield.EVEN_BETTER_TAG.note

'<EVEN_BETTER_TAG>'

Every factory method as well as template has parameters `note` and `active` to set those on creation (note that if multiple datasets within a category has same tag, a running index is appended to keep tags unique).

In [45]:
a5.data.create_bfieldcartesian(
    bxyz=[1.0, 0.0, 0.0]*unyt.T,
    jacobian=np.zeros((3,3))*unyt.T/unyt.m,
    axisrz=[1.0, 0.0]*unyt.m,
    rhoval=1.0,
    note="<EVEN_BETTER_TAG> First magnetic field.",
    activate=True,
)

a5.data.bfield.show_contents()

BfieldCartesian_2 2026-06-23 15:37:05 [active]
EVEN_BETTER_TAG_0
<EVEN_BETTER_TAG> First magnetic field.

BfieldCartesian_1 2026-06-23 15:36:28
MYTAG
<MYTAG> First magnetic field.

BfieldAnalytical_1 2026-06-23 13:33:55
EVEN_BETTER_TAG_1
<EVEN_BETTER_TAG>




## Accessing inputs via the output

The inputs used in a simulation can be accessed from the output by querying the corresponding input category.

In [47]:
bfield_used = a5.data.active.bfield
bfield1 = a5.data.bfield.BfieldAnalytical_1

print(bfield_used is bfield1)


True


Likewise the options are stored within the output (note that you cannot alter the stored options).

In [54]:
print(a5.data.active.options)
a5.data.active.options.simulation.mode = "guiding-center"
print(a5.data.active.options)

SimulationOptions(simulation=Simulation(mode='guiding-center', record_mode=0, timestep=1e-08, enable_adaptive=True, adaptive_tolerance_orbit=1e-08, adaptive_tolerance_collisions=0.1), physics=Physics(enable_orbit_following=True, enable_coulomb_collisions=0, enable_mhd=0, enable_atomic=0, enable_icrh=0, enable_aldforce=0, disable_first_order_gctransformation=0, disable_ccoll_gcenergy=0, disable_ccoll_gcpitch=0, disable_ccoll_gcspatial=0, reverse_time=0), endconditions=Endconditions(activate_simulation_time_limits=True, activate_real_time_limit=0, activate_rho_limit=0, activate_energy_limits=0, activate_wall_hits=0, activate_orbit_limit='no', activate_neutralization=0, activate_ionization=0, lab_time_limit=1.0, max_mileage=0.0001, max_real_time=3600.0, rho_coordinate_limits=(0.0, 1.0), min_energy=1000.0, local_thermal_limit=2.0, max_number_of_toroidal_orbits=100, max_number_of_poloidal_orbits=100), orbit=OrbitParams(collect='no', buffer_size=100, interval=0.0, poloidal_angles=(0.0,), tor

## Storing data on disk

So far all data has been stored in the memory, and as such is lost when the Python session is terminated.

We can save our data to a file at any point.

In [61]:
a5.data.save("ascot.h5")

FileExistsError: The file 'ascot.h5' already exists.

However, there are some rules.
First, if filename is specified when `Ascot` object is created, then by default all data will be stored to disk.
This can be prevented using the `save_hdf5` parameter in factory methods and templates.
Second, saving the output to disk also saves all inputs (otherwise the reproducibility could be lost).
Note that if inputs are not stored to disk, you cannot save the simulation output to disk.

In [ ]:
from a5py.templates import PremadeMagneticField

# Inputs will now be stored to disk by default
nmrk = 1
a5.data.create_guidingcentermarker(
    species="alpha",
    r=(6.2 + (8. - 6.2) * np.random.rand(nmrk))*unyt.m,
    z=0.*unyt.m,
    ekin= 3.5e6*unyt.eV,
    pitch=1. - 2 * np.random.rand(nmrk),
    activate=True, # New inputs are not set as active by default
)

# We can explicitly forbid data from being stored to disk
template = PremadeMagneticField(a5, field="iter-baseline")
template.create_input(activate=True, store_hdf5=True)

# Output won't be stored to disk since not all inputs are
run = a5.simulate(params=opt)

# Storing the output also saves the input
run.save()

## Staging and unstaging

Inputs have to be *staged* before they can be used in simulations or interpolation.
Staging involves constructing splines, setting up data hierarchies and so forth.
As such it can be memory-intensive, which is why the methods that require staged data, unstage it immediately once it is no longer needed.

However, staging can also be time-consuming which is why it is possible to manually stage (and unstage) data using the methods `stage` and `unstage`.
Note that data that is stored in memory is always staged, so staging is only needed for data that is stored on disk.

In [1]:
a5.data.bfield.BfieldAnalytical_1.stage()

NameError: name 'a5' is not defined

## Removing data

Each dataset has a method `destroy` that removes the data also from disk.
To enforce reproducibility, inputs cannot be removed if they are used by any outputs unless the outputs are removed first.

The way how HDF5 format works, the file needs to be *repacked* to free the space previously occupied by the unwanted data.
This can be time-consuming for large files, which is why `destroy` has an optional parameter `repack=True`.
When removing multiple datasets, use `repack=False` for all but the last dataset so that the repacking is done only once when all unwanted data has been removed.

In [ ]:
a5.data.bfield.BfieldAnalytical_1.destroy()

a5.data.bfield.destroy()



## Dryruns

In some rare instances (or more frequently in ASCOT5's internal tools), one might wish to create dataset just to examine it without intention to keep it.
In this case it is unnecessary to include it in the data tree.
All factory methods and templates have parameter `preview=False` which can be toggled to create independent dataset.
Note that this dataset cannot be used to run simulations.

In [ ]:
print("Current contents")
a5.data.bfield.show_contents()

print("Creating a new field without including it to tree")
bdata = a5.data.create_bfieldcartesian("", preview=True)

print("Current contents")
a5.data.bfield.show_contents()

## Export

All datasets can be converted to a dictionary for exporting the data or for tuning some parameters.
The dictionary contains values for the parameters that are used by the factory methods.

In [ ]:
bdata = a5.data.bfield.active.export()

new_file = Ascot()
new_file.data.import_data(bdata)
